# European Flight Delay Prediction — Train/Development/Test splits

This notebook takes the sample of 100,000 previously created from the Flight Delays 2023–2025 Parquet raw dataset from Hugging Face and stored in CGS bucket and creates seperate parquet files for Training, Development and Test.

The sample dataset was created in /notebooks/02_analysis_and_downsampling.ipynb


For reproducability we are capturing and storing the following - **TODO add detail of where these are stored and include in report**
* CGS generation version of sample dataset used
* SHA-256 checksum finger prints for the sample dataset
* Split ID manifests
* Git commit version of this notebook use to create the splits


## Prerequisites

The project dependencies are defined in the repository-level `requirements.txt`
file. When this notebook is opened in Google Colab from GitHub, the repository is cloned into the Colab environment
before installing the required dependencies.

In [2]:
import os

# Clone the project repository into the Colab environment
REPO_URL = "https://github.com/sharonjmaye/european-flight-delay-prediction.git"
REPO_DIR = "european-flight-delay-prediction"
if os.path.exists(REPO_DIR):
    print(f"Repository already exists: {REPO_DIR}")
else:
    !git clone {REPO_URL} {REPO_DIR}
    print("Repository cloned successfully.")

# Change the current working directory to the project root
%cd {REPO_DIR}

Cloning into 'european-flight-delay-prediction'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 30 (delta 10), reused 20 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 562.77 KiB | 14.43 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Repository cloned successfully.
/content/european-flight-delay-prediction


## 1. Setup & Environment Installation

The project dependencies should be listed in `requirements.txt`. Run the following cell before continuing. If packages are newly installed, restart the Jupyter kernel if prompted.

In [4]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 21.0.0
    Uninstalling pyarrow-21.0.0:
      Successfully uninstalled pyarrow-21.0.0


## 2. Configuration & GCP Authentication

Set the project and bucket that the pipeline will use & authenticate to GCP.

Review the folllowing values before running to ensure they are correct for your run:

*   PROJECT_ID
*   BUCKET_NAME
*   SOURCE_GENERATION - leave `None` to capture the current generation at the start of this run, or set it to a previously recorded generation string from the CGS version history to reproduce an earlier run. The notebook downloads the generation explicitly once identified. An old generation can be retrieved only while GCS retains it.
* EXPECTED_SOURCE_SHA256 - If you supply SOURCE_GENERATION and want to verify it has expected SHA-256 fingerprint you should also provide that value here






In [15]:
from google.colab import auth
auth.authenticate_user()

import hashlib
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow as pa
import sklearn
from google.cloud import storage
from sklearn.model_selection import train_test_split, StratifiedKFold

PROJECT_ID = "setu-data-handling"
BUCKET_NAME = "european-flight-delay-prediction"
SOURCE_OBJECT = "processed/flights_downsampled_100k.parquet"
SOURCE_GENERATION = None
EXPECTED_SOURCE_SHA256 = None
RANDOM_STATE = 42
TARGET_COLUMN = "delayed_15min"
N_FOLDS = 8
UPLOAD_TO_GCS = True

WORK_DIR = Path("/content/flight_split_run")
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_SOURCE = WORK_DIR / "flights_downsampled_100k.parquet"

client = storage.Client(project=PROJECT_ID)
bucket = client.bucket(BUCKET_NAME)

print(f"Target project: {PROJECT_ID}")
print(f"Target bucket: gs://{BUCKET_NAME}")

Target project: setu-data-handling
Target bucket: gs://european-flight-delay-prediction


## 3. Download processed dataset
Capture and download the specific version of the processed dataset from CGS based on the SOURCE_GENERATION

In [16]:
if SOURCE_GENERATION is None:
    latest_blob = bucket.blob(SOURCE_OBJECT) # Read latest version if no SOURCE_GENERATION supplied
    latest_blob.reload() # Get this objects metadata
    source_generation = int(latest_blob.generation) # Store its generation number as an integer
else:
    source_generation = int(SOURCE_GENERATION) # Else use generation number supplied

source_blob = bucket.blob(SOURCE_OBJECT, generation=source_generation) # Download the file based with the specific generation version value to local storage
source_blob.download_to_filename(str(LOCAL_SOURCE))

# Calculate a unique SHA-256 digital fingerprint for file at the path specified and return it as 64 character hexadecimal string
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

source_sha256 = sha256_file(LOCAL_SOURCE) # The SHA-256 digital fingerprint generated for the local file will later be saved in our provenance.json file
if EXPECTED_SOURCE_SHA256 is not None:
    assert source_sha256 == EXPECTED_SOURCE_SHA256, (
        "Downloaded file does not match the expected SHA-256."
    )

print("Source SHA-256:", source_sha256)
print("Input:", f"gs://{BUCKET_NAME}/{SOURCE_OBJECT}")
print("Pinned generation:", source_generation)
print("SHA-256:", source_sha256)
print("Downloaded to:", LOCAL_SOURCE)

Source SHA-256: 665605b3fc10bbfce8c7446b7eee9d1ba0cb10e3f379793bc437b2f494f4e69c
Input: gs://european-flight-delay-prediction/processed/flights_downsampled_100k.parquet
Pinned generation: 1791128434018298
SHA-256: 665605b3fc10bbfce8c7446b7eee9d1ba0cb10e3f379793bc437b2f494f4e69c
Downloaded to: /content/flight_split_run/flights_downsampled_100k.parquet


## 4. Load and Validate the processed dataset
Check the following
* It has the correct number of rows
* There are no missing id's
* There are no duplicate id's
* There is no missing values in the TARGET_COLUMN i.e. delayed_15min column and it only contains values 0 or 1

In [17]:
df = pd.read_parquet(LOCAL_SOURCE)
print("Dataset shape:", df.shape)
display(df.head())

assert len(df) == 100_000, f"Expected 100,000 rows, got {len(df):,}"
assert df["id"].notna().all(), "Missing flight IDs"
assert df["id"].is_unique, "Flight IDs are not unique"
assert df[TARGET_COLUMN].notna().all(), "Missing target values"
assert set(df[TARGET_COLUMN].unique()).issubset({0, 1}), "Unexpected target classes"
print("Input validation passed.")

Dataset shape: (100000, 24)


,id,icao24,flt_id,dof,adep,ades,adep_p,ades_p,registration,model,...,last_seen,version,unix_time,source_month,source_year,duration_min,expected_duration_min,delay_min,delayed_15min,route
0,10288345124321242909,320056,DRAG151,2025-05-27,LIPC,LIPE,,,VF-151,None,...,2025-05-27 16:34:00,v2.0.0,1.748362e+09,202505,2025,30.333333,49.833333,-19.500000,0,LIPC-LIPE
1,15072800183822972252,3c65c7,DLH8U,2025-06-29,EGLL,EDDF,,,D-AING,Airbus A320 271NSL,...,2025-06-29 12:07:30,v2.0.0,1.751195e+09,202506,2025,60.500000,65.333333,-4.833333,0,EGLL-EDDF
2,10821715541356530000,4078db,AUR66V,2023-08-23,EGJB,EGNM,,,G-ORAI,None,...,2023-08-23 14:51:15,v2.0.0,1.692798e+09,202308,2023,75.833333,78.083333,-2.250000,0,EGJB-EGNM
3,15423758426971463904,347258,VOE4FC,2025-05-06,LIPX,LEMD,,,EC-NTU,None,...,2025-05-06 17:11:00,v2.0.0,1.746544e+09,202505,2025,125.333333,117.625000,7.708333,0,LIPX-LEMD
4,8484852164953440993,4b161b,SWR2MC,2025-10-09,LSZH,EDDM,,,HB-IJR,A320-214,...,2025-10-09 06:19:05,v0.0.2,NaN,202510,2025,44.666667,43.500000,1.166667,0,LSZH-EDDM


Input validation passed.


## 5. Stratified 80/10/10 split

Stratifying on the TARGET_COLUM i.e `delayed_15min` preserves approximately the same delay rate in each dataset as in the original dataset.

The delay rate of each dataset output at the end should be similar.


Note: This is a random baseline, not a chronological or unseen-airport holdout. Those can be separate later experiments.

In [18]:
# Split dataset into two groups - 80% in train_df and 20% in temp_df using RANDOM_STATE so can be reproduced if necessary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df[TARGET_COLUMN],
)

# Split the temp_df now into dev and test 50/50 so they will each end up with 10%
dev_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df[TARGET_COLUMN],
)

print(f"Splits complete: Train ({len(train_df):,}), Dev ({len(dev_df):,}), Test ({len(test_df):,})")
print("Delay rates:")
display(pd.DataFrame({
    "dataset": ["full", "train", "dev", "test"],
    "rows": [len(df), len(train_df), len(dev_df), len(test_df)],
    "delay_rate_pct": [100*x[TARGET_COLUMN].mean() for x in (df, train_df, dev_df, test_df)] # Calculate the average value of the TARGET_COLUMN and multiply by 100 to get percentage
}).round(3))

Splits complete: Train (80,000), Dev (10,000), Test (10,000)
Delay rates:


,dataset,rows,delay_rate_pct
0,full,100000,8.363
1,train,80000,8.362
2,dev,10000,8.370
3,test,10000,8.360


## 6. Create 8 stratified cross-validation folds from TRAIN

Create 8 stratified cross validation folds from the training data with approximately 10,000 records each.

A model will then be trained with 7 of the folds and validated using the other which it wont yet have seen.

This provides us with the option to try 8 different models using the training data alone.

We save an ID-to-fold manifest so fold assignments can be audited without storing eight duplicate datasets



In [19]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE) # Define fold strategy in this case
cv_folds = {}

# This will be used to store record of which flight Id's were assigned to each fold for reproducability and auditing, it will contain 8 tables with approximately 10,000 rows of validation records along with the fold number
cv_assignment_parts = []

# split the training data into the specified number of folds considering the TARGET_COLUMN, enumerate returns fold numbers from 0 to 7 in this case
# and loop through that number of times using different training/validation combination each time
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df[TARGET_COLUMN])):
    # create two dataframes for the current fold (training and validation) selecting rows based on their numerical positions
    cv_folds[fold_idx] = (
        train_df.iloc[train_idx].copy(),
        train_df.iloc[val_idx].copy(),
    )

    # Extract the unique flight ID's for the 10,000 validation flights for this fold into a NumPy Array and stores these along with the fold number
    # in cv_assignment_parts
    cv_assignment_parts.append(pd.DataFrame({
        "id": train_df.iloc[val_idx]["id"].to_numpy(),
        "validation_fold": fold_idx,
    }))

# Combine the eight smaller cv_assignment_parts DataFrames into one larger DataFrame.
cv_manifest = pd.concat(cv_assignment_parts, ignore_index=True)

# Ensure that all flights from the training dataset have been assigned to a fold for validation and that all the id's are unique
assert len(cv_manifest) == len(train_df)
assert cv_manifest["id"].is_unique
print("Cross-validation folds:", len(cv_folds))
print(cv_manifest.groupby("validation_fold").size())

Cross-validation folds: 8
validation_fold
0    10000
1    10000
2    10000
3    10000
4    10000
5    10000
6    10000
7    10000
dtype: int64


## 7. Verify no records are missing or duplicated between splits


In [20]:
train_ids, dev_ids, test_ids = (set(x["id"]) for x in (train_df, dev_df, test_df))
assert (len(train_df), len(dev_df), len(test_df)) == (80_000, 10_000, 10_000)
assert train_ids.isdisjoint(dev_ids)
assert train_ids.isdisjoint(test_ids)
assert dev_ids.isdisjoint(test_ids)
assert train_ids | dev_ids | test_ids == set(df["id"])
print("All split checks passed.")

All split checks passed.


## 8. Capture the exact Git code version

Record which version of the notebook that was used to create the splits.

For reproducible release,
commit and push this notebook before running then rerun the notebook from that commit.


In [22]:
# Since we already did '%cd' into REPO_DIR, the current directory '.' is our git repository.
git_target_dir = "." if os.path.basename(os.getcwd()) == REPO_DIR else REPO_DIR

git_commit = subprocess.check_output(
    ["git", "-C", str(git_target_dir), "rev-parse", "HEAD"], text=True
).strip()
git_status = subprocess.check_output(
    ["git", "-C", str(git_target_dir), "status", "--porcelain"], text=True
).strip()
print("Git commit:", git_commit)
print("Working tree clean:", not bool(git_status))
if git_status:
    print("WARNING: Uncommitted changes exist; commit them for full code reproducibility.")

Git commit: cc2d2938414166f7c3482ca7dc6cca3a149150fe
Working tree clean: False


In [ ]:
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

LOCAL_PROCESSED_PATH = "/content/flights_downsampled_100k.parquet"

df = pd.read_parquet(LOCAL_PROCESSED_PATH)

print("Dataset shape:", df.shape)
display(df.head())

## 3. Download a specific source object generation and verify IDs

In [ ]:
from datetime import datetime, timezone
import hashlib, json, subprocess
import pandas as pd
import sklearn
import pyarrow
from sklearn.model_selection import train_test_split
from google.cloud import storage

client = storage.Client()
blob = client.bucket(BUCKET_NAME).blob(INPUT_OBJECT)
blob.reload()
source_generation = str(blob.generation)
source_local = OUTPUT_DIR / "source_100k.parquet"
blob.download_to_filename(str(source_local))

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

source_sha256 = sha256_file(source_local)
df = pd.read_parquet(source_local)
assert len(df) == 100_000
assert df["id"].notna().all() and df["id"].is_unique
assert df["delayed_15min"].notna().all()
assert set(df["delayed_15min"].unique()) == {0, 1}
print("Source generation:", source_generation)
print("Source SHA-256:", source_sha256)


## 4. Stratified 80/10/10 split
The second split takes half of the 20% holdout, producing 80% training, 10% development and 10% test. This is a random stratified **baseline**, not a chronological or unseen-airport test.

In [ ]:
train_df, holdout_df = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE,
    stratify=df["delayed_15min"]
)
dev_df, test_df = train_test_split(
    holdout_df, test_size=0.50, random_state=RANDOM_STATE,
    stratify=holdout_df["delayed_15min"]
)
train_df = train_df.reset_index(drop=True)
dev_df = dev_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
assert (len(train_df), len(dev_df), len(test_df)) == (80_000, 10_000, 10_000)
ids = [set(part["id"]) for part in (train_df, dev_df, test_df)]
assert not (ids[0] & ids[1] or ids[0] & ids[2] or ids[1] & ids[2])
assert ids[0] | ids[1] | ids[2] == set(df["id"])
for label, part in [("train",train_df),("development",dev_df),("test",test_df)]:
    print(label, len(part), "delayed %:", round(part["delayed_15min"].mean()*100,3))


## 5. Save split files and a row-ID manifest
Keep all original fields for traceability; exclude leakage fields such as `delay_min`, `first_seen`, `last_seen`, and `duration_min` when constructing model inputs later.

In [ ]:
splits = {"train":train_df, "development":dev_df, "test":test_df}
for name, frame in splits.items():
    frame.to_parquet(OUTPUT_DIR / f"{name}.parquet", index=False)
manifest = pd.concat(
    [frame[["id"]].assign(split=name) for name, frame in splits.items()],
    ignore_index=True
).sort_values("id")
manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)


## 6. Record exact code/data provenance
A Git commit identifies the code; a GCS generation identifies the stored object; SHA-256 identifies the file contents. Do not treat GCS versioning alone as a permanent audit history, since lifecycle rules can delete noncurrent generations.

In [ ]:
def git_commit(repo_dir):
    try:
        return subprocess.check_output(
            ["git", "-C", str(repo_dir), "rev-parse", "HEAD"],
            text=True, stderr=subprocess.DEVNULL
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None

def git_dirty(repo_dir):
    try:
        return bool(subprocess.check_output(
            ["git", "-C", str(repo_dir), "status", "--porcelain"],
            text=True, stderr=subprocess.DEVNULL
        ).strip())
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None

provenance = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_gcs_uri": f"gs://{BUCKET_NAME}/{INPUT_OBJECT}",
    "input_gcs_generation": source_generation,
    "input_sha256": source_sha256,
    "processing_notebook": "notebooks/03_data_splitting.ipynb",
    "git_commit": git_commit(REPO_DIR),
    "git_worktree_dirty": git_dirty(REPO_DIR),
    "method": "two-stage stratified random split by delayed_15min",
    "random_seed": RANDOM_STATE,
    "split_sizes": {name: len(frame) for name, frame in splits.items()},
    "python_packages": {"pandas":pd.__version__, "scikit_learn":sklearn.__version__, "pyarrow":pyarrow.__version__},
    "outputs": {p.name:sha256_file(p) for p in OUTPUT_DIR.glob("*.parquet")},
    "manifest_sha256": sha256_file(OUTPUT_DIR / "split_manifest.csv")
}
with open(OUTPUT_DIR / "provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)
print(json.dumps(provenance, indent=2))
if provenance["git_commit"] is None or provenance["git_worktree_dirty"]:
    print("WARNING: Run from a clean committed Git checkout for a fully traceable code version.")


## 7. Optional GCS upload and generation receipt
Uploads are disabled by default. The bucket is public read-only to reviewers; uploading requires credentials with write permission. The receipt stores the generation of each uploaded object.

In [ ]:
if UPLOAD_TO_GCS:
    receipts = {}
    for local_path in sorted(OUTPUT_DIR.glob("*")):
        if not local_path.is_file():
            continue
        destination = f"{OUTPUT_PREFIX}/{local_path.name}"
        obj = client.bucket(BUCKET_NAME).blob(destination)
        obj.upload_from_filename(str(local_path), if_generation_match=0)
        receipts[destination] = str(obj.generation)
    receipt_path = OUTPUT_DIR / "gcs_upload_receipt.json"
    receipt_path.write_text(json.dumps(receipts, indent=2))
    print("Uploaded files; generations recorded in", receipt_path)
else:
    print("Upload skipped. Set UPLOAD_TO_GCS=True when ready and authorised.")
